# DR viral content

using geNomad: https://github.com/apcamargo/genomad

In [ ]:
## INSTALLATION
#login5 tmux session
salloc -p cpu -t 4:00:00 -c 4 --mem=5G
module load conda/latest
conda create --prefix /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/genomad
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/genomad
conda install bioconda::genomad
genomad download-database .
#/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/genomad/genomad_db

In [ ]:
#path to co-assembly files: /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/assemblies

In [ ]:
#starting with MCAV co-assembly first, running in an interactive session first (see if 5G is enough)
genomad end-to-end --cleanup --splits 8 /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/assemblies/mcav.contigs-fixed.fsa \
mcav_genomad_output ./genomad_db
#splits might not be necessary with this amount of RAM and MEM - will keep out when running in bash

script to run the rest

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=10G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 6:00:00  # Job time limit
#SBATCH --qos=long #if job takes longer than 48 hrs
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-genomad-run-%j.out  # %j = job ID


module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/genomad

cd /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/genomad

#run for each species
genomad end-to-end --cleanup --splits 8 /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/assemblies/ssid.contigs-fixed.fsa \
ssid_genomad_output ./genomad_db

genomad end-to-end --cleanup --splits 8 /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/assemblies/dcyl.contigs-fixed.fsa \
dcyl_genomad_output ./genomad_db

genomad end-to-end --cleanup --splits 8 /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/assemblies/mmea.contigs-fixed.fsa \
mmea_genomad_output ./genomad_db

genomad end-to-end --cleanup --splits 8 /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/assemblies/dlab.contigs-fixed.fsa \
dlab_genomad_output ./genomad_db

genomad end-to-end --cleanup --splits 8 /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/assemblies/pstr.contigs-fixed.fsa \
pstr_genomad_output ./genomad_db

conda deactivate

# JOB-ID: 64520426
# bash script file name: DR/bash_scripts/DR_genomad.sh

for some reason this script didn't run past the annotate step - python issue. Didn't run into this issue when running it in the interactive session, will do so. The other option is that I do need to include --splits (I didn't initially in the bash script)

I ran above individually, did not take time to troubleshoot.

## CheckV to assess completeness

In [ ]:
## INSTALLATION
module load conda/latest
conda create --prefix /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/checkv -c conda-forge -c bioconda checkv
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/checkv
#downloaded database to scratch directory although it's only like 6GB in size
checkv download_database /scratch4/workspace/nikea_ulrich_uml_edu-DR_data/mapping/checkV_db
export CHECKVDB=/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/checkV_db/checkv-db-v1.5

In [ ]:
#run (individually) it's super quick so fine to do in an salloc session
tmux session login5 (checkv)
salloc -p cpu -t 4:00:00 -c 4 --mem=5G

cd /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/genomad/mcav_genomad_output
checkv end_to_end mcav.contigs-fixed_summary/mcav.contigs-fixed_virus.fna \
-d /scratch4/workspace/nikea_ulrich_uml_edu-DR_data/checkV_db/checkv-db-v1.5 ./checkv_output -t 16

# ran this for each coral species

## Mapping

map the complete viruses back on to samples to check their abundances. Are they found in one particular sample type or found across all samples? or just one sample?

move all virus .fna files from the checkv output to the scratch working path

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=70G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long # extend time limit to longer than 2 days
#SBATCH -t 4:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-mcav-viral-mapping-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8

SAMPLENAME="mcav"
READSPATH="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/DR_final_filtered/${SAMPLENAME}/repaired"
#VCONTIGPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/genomad/${SAMPLENAME}_genomad_output/checkv_output"
VCONTIGFILE="${SAMPLENAME}_viruses.fna"
WORKPATH="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/v_mapping/${SAMPLENAME}"
#mkdir -p "$WORKPATH"
XTRAFILES="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/v_mapping/sams"
mkdir -p "$XTRAFILES"
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/ID_tables"
SAMPLELIST="DR_${SAMPLENAME}_sampleids.txt" 

cd $WORKPATH
#this builds an index of your contigs, which only needs to happen once
bowtie2-build $VCONTIGFILE "$SAMPLENAME"_viral_contigs
# will not accept path before contigs file - must be in the correct dir 
while IFS= read -r SAMPLEID; do
    #align reads to your contigs and collects that in a .sam file
    bowtie2 --threads 11 -x "$SAMPLENAME"_viral_contigs -1 $READSPATH/"${SAMPLEID}"_R1_ready.fastq.gz -2 $READSPATH/"${SAMPLEID}"_R2_ready.fastq.gz -S $XTRAFILES/"${SAMPLEID}".sam
    #make sure to point it to the index not the FIXEDCON file (-x parameter)
    
    #converts your sam file to a bam file, but its neither sorted nor indexed, so we use an Anvi'O script to do so:
    samtools view -F 4 -b -S $XTRAFILES/"${SAMPLEID}".sam -o $WORKPATH/"${SAMPLEID}"-RAW.bam
   
    #index and sort your bam file
    anvi-init-bam $WORKPATH/"${SAMPLEID}"-RAW.bam -o $WORKPATH/"${SAMPLEID}".bam
    
    rm $WORKPATH/"${SAMPLEID}"-RAW.bam
done < "$LISTPATH/${SAMPLELIST}"
echo "Virus mapping success!"

#JOB ID: 64728243
#bash script: DR_mcav_virus_mapping.sh

**other job IDs:** \
ssid: 64729348 \
dcyl: 64729482 \
mmea: 64729856 \
dlab: 64730007 \
pstr: 64730008

### CoverM to look at relative abundance across samples
https://wwood.github.io/CoverM/coverm-genome.html

coverm has already been installed in the anvio-8 environment

In [ ]:
login5 tumx checkv
salloc -p cpu -t 3:00:00 -c 4 --mem=5G
module load conda/latest
conda activate anvio-8

In [ ]:
cd /scratch4/workspace/nikea_ulrich_uml_edu-DR_data/v_mapping/mcav
coverm contig -b ./*.bam -m mean --min-covered-fraction 0 -o ./coverm_mean.tsv
#also do -m covered_fraction to get an idea of coverage